In [1]:
from langsmith import traceable
from src.retrival_methods import retrival_pipeline

import re

@traceable(name="rag_pipeline")
def main_retrival_pipelines(inputs: dict) -> dict:
    query = inputs["question"]
    collection = inputs.get("collection", "DemoRAG")
    pipeline = retrival_pipeline(collection=collection)
    all_retrieval_results = pipeline.multiquery_RRM(query)
    fused_results = pipeline.reciprocal_rank_fusion(
        all_retrieval_results, k=60, verbose=False
    )
    reranked_docs_c = pipeline.reranker_chunks()
    response = pipeline.generate_final_answer(chunks=reranked_docs_c, query=query)

    response_str = str(response)
    thinking_match = re.search(r"<think>(.*?)</think>", response_str, re.DOTALL)
    thinking = thinking_match.group(1).strip() if thinking_match else None
    clean_answer = re.sub(r"<think>.*?</think>", "", response_str, flags=re.DOTALL).strip()
    contexts= [getattr(d, "page_content", str(d)) for d in reranked_docs_c]
    print(f"contexts:{contexts}")
    print(f"type:{type(contexts)}")
    print(f"thinking: {thinking}")
    print(f"answer:{clean_answer}")
    return {
        "answer": clean_answer,
        "context": [getattr(d, "page_content", str(d)) for d in reranked_docs_c],
        "thinking": thinking,  # optional, useful for debugging in the LangSmith UI
    }

In [2]:
import os
from langsmith import Client
from dotenv import load_dotenv
load_dotenv()
client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

try:   
    dataset = client.create_dataset(
        dataset_name="rag-eval-golden-setsing",
        description="Golden QA pairs for RAG evaluation"
    )
    examples = [
    {"question": "What is the refund policy?", "answer": "Refunds are issued within 14 days of purchase."},
    {"question": "How do I reset my password?", "answer": "Go to Settings > Security > Reset Password."},
    # ... add more, ideally pulled from real user logs / support tickets
    ]

    client.create_examples(
        inputs=[{"question": e["question"]} for e in examples],
        outputs=[{"answer": e["answer"]} for e in examples],
        dataset_id=dataset.id,
    )
except Exception as e:
    dataset = client.read_dataset(dataset_name="rag-eval-golden-setsing")




In [3]:
from src.models import build_llms
# imports langchain openai compatiable models
grok_llm, openrouter,local_vllm = build_llms()

judge = local_vllm.with_fallbacks([openrouter,local_vllm])

def _score_from_response(text:str) -> float:
    try:
        return max(0.0, min(1.0,float(text.strip())))
    except ValueError:
        return 0.0

def correctness(inputs:dict, outputs:dict,reference_outputs:dict)-> dict:
    prompt =f"""score 0-1 how factually consistent the generated answer is with the reference answer. Respect with a Only a number.
    QUESTION:{inputs["question"]}
    REFERENCE ANSWER :{reference_outputs["answer"]}
    GENERATED ANSWER :{outputs["answer"]}
    
    """
    score = _score_from_response(judge.invoke(prompt).content)
    return {"key":"correctness", "score": score}

def faithfulness(inputs:dict, outputs:dict, reference_outputs:dict) -> dict:
    context ="\n\n".join(outputs.get("context",[]))
    prompt = f"""score 0-1 how much of the answer's claims are directly support by the context. Respond with Only a number.
    
    CONTEXT:
    {context}
    ANSWER:
    {outputs["answer"]}"""
    score = _score_from_response(judge.invoke(prompt).content)
    return {"key":"faithfulness","score":score} 

def context_relevance(inputs:dict,outputs:dict, reference_outputs:dict) -> dict:
    context ="\n\n".join(outputs.get("context",[]))
    prompt = f"""Score 0-1 how relevant this retrived context is to answering the question. Respond with only a number.
    QUESTION {inputs["question"]}
    CONTEXT:
    {context}
    """
    score= _score_from_response(judge.invoke(prompt).content)
    return {"key": "context_relvance", "score":score}

In [ ]:
from langsmith.evaluation import evaluate

results = evaluate(
    main_retrival_pipelines,
    data = "rag-eval-golden-setsing",
    evaluators=[correctness, faithfulness, context_relevance],
    experiment_prefix ="rag-v1",
    max_concurrency=2,
)

View the evaluation results for experiment: 'rag-v1-0de0f105' at:
https://smith.langchain.com/o/bd818278-7588-4f06-af2b-10b8327880c6/datasets/a7bd2f25-1b62-4a77-881e-28a036d38957/compare?selectedSessions=b92b70c7-109e-4dd7-8c8a-decbd68f899d




0it [00:00, ?it/s]

2026-08-02 00:51:50.553 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'ThreadPoolExecutor-3_0': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-02 00:51:50.554 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'ThreadPoolExecutor-3_1': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-02 00:52:12.118 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'ThreadPoolExecutor-3_1': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-08-02 00:52:14.634 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'ThreadPoolExecutor-3_0': missing ScriptRunContext! This warning can be ignored when running in bare mode.


-----Hybrid Retriever intialised

=====Results for Query 1:Steps to recover a forgotten password===
-----Hybrid Retriever intialised

=====Results for Query 1:What is the the refund policy?===
Retrived 7 documents:

Document1:
Hydration reaction starts as soon as water meets the cement particles. Proper curing for a minimum of 7 to 10 days is vital to prevent early moisture 
Document2:
<think>
Okay, let's tackle this query. The user wants a searchable description for document content retrieval. The content provided includes both text
Document3:
Department Heads & Key Contact Information
Document4:
Moreover, stock put-away is guided by the Warehouse Management System (WMS), which assigns specific bin locations based on item weight, safety classif
Document5:
Furthermore, the microservices architecture employs Kong API Gateway for routing client calls, securing endpoints via OAuth 2.0, and applying rate lim
Document6:
<think>
Okay, let's tackle this query. The user wants a searchable desc

In [ ]:
from langsmith.evaluation import evaluate

results = evaluate(
    main_retrival_pipelines,
    data = "rag-eval-golden-sets",
    evaluators=[correctness, faithfulness, context_relevance],
    experiment_prefix ="rag-v1",
    max_concurrency=2,
)